# Air Quality Index Probability Prediction (MLOps Version)
*How can Mixture Density Networks improve air pollution forecasting by providing uncertainty-aware predictions that support more informed and reliable business or policy decisions?*

Traditional air quality forecasting models frequently provide deterministic, single-point predictions without quantifying the associated uncertainty. This limitation can be problematic, especially when decision-making requires an understanding of the range of possible outcomes. Recent studies have highlighted this issue. For instance, research has shown that most current data-driven air quality forecasting solutions lack proper quantifications of model uncertainty, which is crucial for communicating the confidence in forecasts. This gap underscores the need for models that can provide probabilistic forecasts, offering a distribution of possible outcomes rather than a single deterministic prediction.

Incorporating uncertainty quantification into air quality forecasts allows for better risk assessment and more informed decision-making. Probabilistic models, such as those using deep learning techniques, have been developed to address this need, providing more reliable uncertainty estimates and improving the practical applicability of air quality forecasts.

The goal of this project is to develop a probabilistic air quality forecasting model that captures a full range of possible pollutant concentrations, rather than relying on single-point predictions.  Use Mixture Density Networks (MDNs) to model predictive uncertainty.

Train and evaluate the MDN framework using various sequence modeling architectures:
- LSTM-MDN
- GRU-MDN
- Classic RNN-MDN
- TCN-MDN
- Transformer-MDN


## System Overview

The system is composed of several containerized services that together enable data ingestion, preprocessing, model training, experiment tracking, drift monitoring, and API-based model serving:
- Airflow orchestrates the machine learning pipelines. It schedules and manages tasks such as data ingestion, preprocessing, model training, drift detection, and promotion of the best model to production.
- FastAPI exposes the trained model as a REST API. It provides endpoints for prediction (/predict), retrieving model information (/model), and reloading the production model (/reload_model).
- MLflow tracks experiments, hyperparameters, and metrics. It also manages the model registry, which stores multiple model versions and defines the “Production” model used by FastAPI.
- Postgres serves as the metadata store for both Airflow and MLflow, ensuring experiment and pipeline metadata are persisted.
- Redis supports Airflow’s CeleryExecutor for distributed task execution and result backend management.
- Optuna is used for hyperparameter optimization, with studies persisted in Postgres. Initialization scripts ensure database setup for Optuna and MLflow.
- Evidently is integrated within Airflow tasks to monitor data drift and prediction drift, producing reports stored as artifacts for later inspection.

Together, these components form a reproducible, automated ML system that moves from raw air quality data to production-ready predictive models served through an API.


## Setup Instructions
> Notes for Methods 1 and 2:
> - CUDA GPU support in highly recommended for model training.
> - Apple Silicon MPS will never be supported in this Dockerized setup.

This method runs the entire pipeline using **Apache Airflow** in a Dockerized environment. It is the recommended way to orchestrate scheduled training, evaluation, and report generation.

#### 1. Install Docker Compose

Make sure you have the following installed:
- [Docker](https://docs.docker.com/get-docker/)
- [Docker Compose](https://docs.docker.com/compose/install/)
  (v2 preferred: `docker compose` instead of `docker-compose`)

To verify installation:
```bash
docker --version
docker compose version
```

#### 2. Clone the repository
Clone the repository to your local machine:
```bash
git clone https://github.com/PeteCastle/6bd14ff485084ce2fd9c18e9539cab76bc04816b587a434afc54c644bc3abdc7_aqi_probability_prediction aqi-probability-prediction
```

#### 3. Setup the Airflow environment variables
- Navigate to the project directory.
- Inside the `config` directory, create an `.env` file using `config/.env.example` as a reference.
- Inside the `config` directory, create an `airflow.cfg` file using `config/airflow.cfg.example` as a reference.
- Inside the `config` directory, create a `config.yaml` file using `config/config.yaml.example` as a reference.

#### 4. Build the services
Build all services defined in the Docker Compose file:
```bash
docker compose -f docker/docker-compose.yml build
```

#### 5. Start the Airflow services
Start the Airflow webserver, scheduler, and other services in detached mode:
```bash
docker compose -f docker/docker-compose.yml up -d
```

#### 6. Running the Pipeline
Access the Airflow web interface and DAGs at `http://localhost:8080/dags`.
![Airflow DAGs](../docs/assets/airflow_dags.png)
Click on `training_dag` to view the DAG details.

![Model Training Pipeline](../docs/assets/training_dag.png)
Click on `Trigger` to run the pipeline manually
![Pipeline Trigger](../docs/assets/pipeline_trigger.png)
Modify parameters if you want to specify the number of trials, epochs, or in dry run.  The script will always generate a report after training and evaluation.

#### Other Pipelines
- `drift_dag`: Detects data drift using Evidently and generates a report.
- `promote_model_dag`: Promotes the best model to production based on evaluation metrics.

#### 7.  Test the Datasets
To test the datasets, you should create an environment first.
```bash
uv venv
source .venv/bin/activate

uv pip install -r pyproject.toml
```

Run the following command on the root directory:

```bash
pytest
```


## Model Deployment
### Prediction Request

In [ ]:
import requests
import json

# Endpoint
url = "http://3.226.46.78:8000/predict"

# Request body
payload = [
    {
        "co": 134242342,
        "no": 2233,
        "no2": 23233,
        "o3": 232324,
        "so2": 53232,
        "pm2_5": 323236,
        "pm10": 232327,
        "nh3": 83232323,
        "city": "Navotas",
    },
    {
        "co": 1,
        "no": 2,
        "no2": 3,
        "o3": 4,
        "so2": 5,
        "pm2_5": 6,
        "pm10": 7,
        "nh3": 8,
        "city": "Valenzuela",
    },
]

# Make POST request
try:
    response = requests.post(url, json=payload, timeout=15)
    response.raise_for_status()  # Raise an error for bad responses (4xx/5xx)
    print("✅ Response:")
    print(json.dumps(response.json(), indent=4))
except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")

### Model Info

In [ ]:
import requests
import json

# Endpoint
url = "http://3.226.46.78:8000/model"

# Make GET request
try:
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    print("✅ Model Info:")
    print(json.dumps(response.json(), indent=4))
except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")

### Reload Model

In [ ]:
import requests
import json

# Endpoint
url = "http://3.226.46.78:8000/reload_model"

# Make POST request (no body required)
try:
    response = requests.post(url, timeout=15)
    response.raise_for_status()
    print("✅ Reload response:")
    print(json.dumps(response.json(), indent=4))
except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")

## Drift Detection

In [ ]:
import json
from src.constants import OUTPUT_DIR

# Path to the drift report JSON
drift_report_path = OUTPUT_DIR / "drift_report.json"

# Load the drift report
with open(drift_report_path, "r") as f:
    drift_report = json.load(f)

# Print nicely formatted drift report
print("✅ Drift Report Summary")
print(json.dumps(drift_report, indent=4))

No significant drift was detected (drift_detected: false), meaning the current data distribution is still consistent with the reference dataset. The top features with minor drift signals are f211, f1450, and f1134, but their drift scores are very low (~0.03), well below any concerning threshold. The overall drift score is 0.0057, which confirms data stability and suggests no immediate need for retraining or intervention.